# Car Price EDA and Model Selection

Notebook dokumentuje eksploracje danych, wybor cech oraz porownanie modeli dla aplikacji wyceny aut uzywanych.

## Setup

Notebook sprawdza, czy dane surowe sa dostepne w `data/raw/Car_sale_ads.csv`. Jesli pliku nie ma, komorka przed wczytaniem danych pobierze dataset z Kaggle przez produkcyjna funkcje `download_dataset()`. Do pobrania wymagany jest `KAGGLE_API_TOKEN` w lokalnym `.env` albo token zapisany w konfiguracji Kaggle.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from car_price_prediction import config
from car_price_prediction.data.download import download_dataset
from car_price_prediction.data.preprocessing import preprocess_data
from car_price_prediction.feature_options import save_feature_options
from car_price_prediction.model.train import (
    aggregate_feature_importance,
    evaluate_candidates,
    model_feature_importance,
    save_model_artifacts,
    select_model,
    train_final_pipeline,
    vehicle_age_reference_year_for_training,
)

sns.set_theme(style="whitegrid")


## Load raw data

In [ ]:
if config.RAW_DATA_PATH.exists():
    print(f"Raw dataset already exists: {config.RAW_DATA_PATH}")
else:
    print("Raw dataset is missing. Downloading from Kaggle...")
    download_dataset()


In [ ]:
raw = pd.read_csv(config.RAW_DATA_PATH)
raw.shape

In [ ]:
raw.head()

## Dataset overview

In [ ]:
overview = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": raw.isna().mean().round(3),
    "unique": raw.nunique(dropna=True),
})
overview.sort_values("missing_pct", ascending=False).head(30)

In [ ]:
raw[config.CURRENCY_COLUMN].value_counts(dropna=False).head(10)

## Preprocess with production code

Ten krok uzywa tego samego preprocessingu, ktory jest uruchamiany przez `uv run preprocess-data`. Preprocessing czysci podejrzane wartosci cech, usuwa rekordy z niewiarygodnym targetem, usuwa dokladne duplikaty oraz tworzy `Vehicle_age_years` jako `max(Production_year z datasetu) - Production_year`. Niewiarygodny target obejmuje m.in. mlode auta z cena ponizej 1000 PLN oraz skrajne ceny powyzej 3 mln PLN. Formularz aplikacji nadal powinien prosic uzytkownika o rok produkcji; wiek auta jest cecha techniczna liczona wewnatrz pipeline.


In [ ]:
data = preprocess_data(raw)
data.shape

In [ ]:
data[config.MODEL_COLUMNS].describe(include="all").T

## Target analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data[config.TARGET_COLUMN], bins=80, ax=axes[0])
axes[0].set_title("Price distribution")
sns.histplot(data[config.TARGET_COLUMN].clip(upper=data[config.TARGET_COLUMN].quantile(0.99)), bins=80, ax=axes[1])
axes[1].set_title("Price distribution clipped at p99")
plt.tight_layout()

In [ ]:
data[config.TARGET_COLUMN].quantile([0.01, 0.05, 0.5, 0.95, 0.99]).to_frame("price_pln")

## Feature review

### Mileage distribution and outliers

W surowym `Mileage_km` widac pojedyncze nielogiczne wartosci, np. przebiegi rzedu wielu milionow albo setek milionow kilometrow. To wyglada na bledy wprowadzania danych, nie realne cechy aut. Z tego powodu produkcyjny preprocessing nie przycina tych wartosci do limitu, tylko ustawia je jako brak danych (`NaN`). Dzieki temu nie tracimy calego wiersza, a model moze imputowac brakujacy przebieg na podstawie danych treningowych. Obserwacja jest wazna rowniez przy interpretacji korelacji: Pearson na surowych wartosciach jest bardzo wrazliwy na takie outliery.


In [ ]:
raw_mileage = pd.to_numeric(raw["Mileage_km"], errors="coerce")
processed_mileage = data["Mileage_km"]

mileage_quantiles = [0.01, 0.05, 0.5, 0.95, 0.99, 0.999, 1.0]
mileage_summary = pd.DataFrame({
    "raw_mileage_km": raw_mileage.quantile(mileage_quantiles),
    "processed_mileage_km": processed_mileage.quantile(mileage_quantiles),
})
mileage_summary.loc["missing_count"] = [raw_mileage.isna().sum(), processed_mileage.isna().sum()]
mileage_summary.loc["above_max_count"] = [
    (raw_mileage > config.MAX_MILEAGE_KM).sum(),
    (processed_mileage > config.MAX_MILEAGE_KM).sum(),
]
mileage_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

raw_plot = raw_mileage.dropna()
raw_p99 = raw_plot.quantile(0.99)
sns.histplot(raw_plot.clip(upper=raw_p99), bins=80, ax=axes[0])
axes[0].set_title(f"Raw mileage clipped at p99 ({raw_p99:,.0f} km)")
axes[0].set_xlabel("Mileage_km")

processed_plot = processed_mileage.dropna()
sns.histplot(processed_plot, bins=80, ax=axes[1])
axes[1].set_title("Processed mileage after nullifying unreliable values")
axes[1].set_xlabel("Mileage_km")

plt.tight_layout()


In [ ]:
raw.loc[raw_mileage > config.MAX_MILEAGE_KM, [
    "Price",
    "Condition",
    "Vehicle_brand",
    "Production_year",
    "Mileage_km",
]].sort_values("Mileage_km", ascending=False).head(10)


In [ ]:
data[config.NUMERIC_FEATURE_COLUMNS + [config.TARGET_COLUMN]].corr(numeric_only=True)[config.TARGET_COLUMN].sort_values(ascending=False)

In [ ]:
category_summary = []
for column in config.CATEGORICAL_FEATURE_COLUMNS:
    category_summary.append({
        "column": column,
        "unique": data[column].nunique(dropna=True),
        "top_values": data[column].value_counts(dropna=False).head(8).to_dict(),
    })
pd.DataFrame(category_summary)

## Model comparison

Porownanie uzywa tej samej logiki co produkcyjny skrypt `uv run train-model`: baseline, modele klasyczne i LightGBM. Regula wyboru preferuje LightGBM, jesli bije baseline i jest w granicy 5% najlepszego RMSE.

In [ ]:
results = evaluate_candidates(data)
results_table = pd.DataFrame([r.__dict__ for r in results])
results_table

In [ ]:
selected_model = select_model(results)
selected_model

## Final model artifact and interpretation

Ten krok trenuje wybrany pipeline na calym oczyszczonym zbiorze i zapisuje artefakty tym samym mechanizmem co produkcyjny skrypt `uv run train-model`. Interpretacja waznosci cech jest model-specific: dla LightGBM i modeli drzewiastych jest to `feature_importances_`, a dla modeli liniowych absolutna wartosc wspolczynnika. Nie nalezy traktowac tego jako dowodu zwiazku przyczynowego.


In [ ]:
final_pipeline = train_final_pipeline(data, selected_model)
save_model_artifacts(
    pipeline=final_pipeline,
    results=results,
    selected_model_name=selected_model,
    rows_count=len(data),
    vehicle_age_reference_year=vehicle_age_reference_year_for_training(data),
)
save_feature_options(data)

pd.DataFrame(
    [
        {"artifact": "model", "path": str(config.MODEL_PATH.relative_to(project_root))},
        {"artifact": "metadata", "path": str(config.MODEL_METADATA_PATH.relative_to(project_root))},
        {"artifact": "metrics", "path": str(config.TRAINING_METRICS_PATH.relative_to(project_root))},
        {"artifact": "feature_options", "path": str(config.FEATURE_OPTIONS_PATH.relative_to(project_root))},
    ]
)


### Feature importance by source field

Ta tabela agreguje waznosci po oryginalnych polach formularza. Jest najczytelniejsza do raportu, bo kategorie po one-hot encodingu sa zwiniete z powrotem do kolumn typu `Vehicle_brand`, `Fuel_type` albo `Type`.


In [ ]:
source_importance = aggregate_feature_importance(final_pipeline)
source_importance


In [ ]:
if source_importance.empty:
    print("Selected model does not expose feature importance or coefficients.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=source_importance.head(15),
        x="importance_share",
        y="source_feature",
        ax=ax,
    )
    ax.set_title("Aggregated feature importance of the selected model")
    ax.set_xlabel("Importance share")
    ax.set_ylabel("")
    plt.tight_layout()


### Top transformed features

Ta tabela pokazuje juz cechy po transformacji pipelineu, wiec pola kategoryczne sa rozbite na konkretne wartosci, np. marka lub typ paliwa.


In [ ]:
encoded_importance = model_feature_importance(final_pipeline)
encoded_importance.head(30)


## Notes for final report

- Target: cena ofertowa auta w PLN.
- Input aplikacji: uzytkownik wpisuje rok produkcji, a pipeline przelicza go na wiek auta.
- Features modelowe: stan, marka, wiek auta w latach, przebieg, moc, pojemnosc, paliwo, naped, skrzynia, typ nadwozia, liczba drzwi.
- Notebook zapisuje model tym samym writerem co skrypt `uv run train-model`; do powtarzalnego treningu produkcyjnego nadal preferowany jest skrypt CLI.
- Mileage outliers: nielogiczne przebiegi poza zakresem 0-1 000 000 km sa traktowane jako brak danych, nie jako realna wartosc ani powod usuniecia calego wiersza.
- Exact duplicates: po czyszczeniu usuwane sa tylko rekordy identyczne po wszystkich kolumnach processed, lacznie z targetem.
- Target outliers: rekordy z cena powyzej 3 mln PLN sa odrzucane jako niewiarygodne dla zakresu aplikacji.
